# 🎙️ Estudio de voz en Google Colab

Este cuaderno arranca tu Estudio de voz con la **voz de verdad** (Chatterbox Multilingüe, licencia MIT) usando una GPU gratis de Google.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** y Guardar.

Luego pulsa **Entorno de ejecución → Ejecutar todo** (o Ctrl + F9). La primera vez tarda unos 5–10 minutos en instalar y descargar el modelo. Al final aparece un enlace para abrir el estudio.

In [ ]:
#@title 1. Comprobar la GPU
import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ No hay GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4, y vuelve a ejecutar.")
print("✅", gpu)

In [ ]:
#@title 2. Guardar tus voces y audios en Google Drive
#@markdown Si lo dejas marcado, tus voces clonadas y tus audios se guardan en **Mi unidad/EstudioDeVoz** y no se pierden al cerrar Colab. Te pedirá permiso para usar tu Drive.
usar_drive = True #@param {type:"boolean"}
import os
if usar_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    DATOS = "/content/drive/MyDrive/EstudioDeVoz"
else:
    DATOS = "/content/estudio-datos"
os.makedirs(DATOS, exist_ok=True)
print("📁 Datos en:", DATOS)

In [ ]:
#@title 3. Instalar el estudio y el motor de voz (unos minutos)
import os, subprocess
if os.path.isdir("/content/estudio-de-voz/.git"):
    subprocess.run(["git", "-C", "/content/estudio-de-voz", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/Jonragitt/estudio-de-voz", "/content/estudio-de-voz"], check=True)
r = subprocess.run("pip install -q chatterbox-tts fastapi uvicorn python-multipart", shell=True, capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
print("✅ Instalado")

In [ ]:
#@title 4. Arrancar el estudio (la primera vez descarga el modelo, unos 3 GB)
import os, subprocess, time, urllib.request, json
subprocess.run("pkill -f 'uvicorn server.main'", shell=True)
env = dict(os.environ, ESTUDIO_DATOS=DATOS)
log = open("/content/estudio.log", "w")
srv = subprocess.Popen(["python", "-m", "uvicorn", "server.main:app", "--host", "0.0.0.0", "--port", "8000"],
                       cwd="/content/estudio-de-voz", env=env, stdout=log, stderr=subprocess.STDOUT)
estado = None
for i in range(180):  # hasta 15 minutos
    time.sleep(5)
    if srv.poll() is not None:
        break
    try:
        estado = json.load(urllib.request.urlopen("http://localhost:8000/api/estado", timeout=600))
        break
    except Exception:
        if i % 6 == 0:
            print("⏳ Cargando…", flush=True)
if not estado:
    print(open("/content/estudio.log").read()[-3000:])
    raise SystemExit("❌ El estudio no arrancó. Mira el registro de arriba.")
if estado["motor"] != "chatterbox":
    print(open("/content/estudio.log").read()[-3000:])
    print("⚠️ Arrancó en modo demo (sin voz de verdad). Mira el registro de arriba.")
else:
    print("✅ Motor de voz listo:", estado)

In [ ]:
#@title 5. Abrir el estudio
#@markdown Abre el enlace que aparece abajo. Funciona mientras esta pestaña de Colab siga abierta.
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8000)")
from IPython.display import HTML, display
display(HTML(f'<a href="{url}" target="_blank" style="font-size:20px">🎙️ Abrir Estudio de voz</a>'))
print(url)

In [ ]:
#@title 6. (Opcional) Enlace para abrirlo desde el móvil u otro ordenador
#@markdown Crea un enlace público temporal con Cloudflare. Cualquiera que tenga el enlace puede usar tu estudio mientras esté encendido, así que no lo compartas.
import subprocess, re, time, os
if not os.path.exists("/content/cloudflared"):
    subprocess.run("wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared", shell=True, check=True)
cf = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in cf.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("🌍 Enlace público:", m.group(0))
        break

## Cuando termines
Cierra esta pestaña o pulsa **Entorno de ejecución → Desconectar y eliminar entorno de ejecución** para liberar la GPU. Google da unas horas de GPU gratis al día; si se acaban, espera al día siguiente.

Para volver otro día, abre este cuaderno y **Ejecutar todo** otra vez. Tus voces siguen en tu Drive.